In [2]:
# Import necessary libraries for data manipulation and LightGBM
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Load the F1 Strategy Dataset
df = pd.read_csv('f1_strategy_dataset_v4.csv')

# Display the first few rows
df.head()

In [4]:
# Basic statistics
df.info()


In [5]:
df.describe()

In [6]:

# Target distribution
df['PitNextLap'].value_counts()
df['PitNextLap'].value_counts(normalize=True)



In [7]:
# Missing values
df.isnull().sum()

In [8]:
# Time-series aware train/test split
# Each unique combination of Driver + Race + Year forms a sequence ordered by LapNumber
# We split each sequence at 70/30 to preserve temporal order

# Group by driver-race-year to identify sequences
sequences = df.groupby(['Driver', 'Race', 'Year'])

train_sequences = []
test_sequences = []

for name, group in sequences:
    # Split each sequence at 70/30
    split_idx = int(len(group) * 0.7)
    train_sequences.append(group.iloc[:split_idx])
    test_sequences.append(group.iloc[split_idx:])

# Concatenate all train and test sequences
train_df = pd.concat(train_sequences, ignore_index=True)
test_df = pd.concat(test_sequences, ignore_index=True)

print(f"Total rows: {len(df)}")
print(f"Train size: {len(train_df)} ({len(train_df)/len(df):.2%})")
print(f"Test size: {len(test_df)} ({len(test_df)/len(df):.2%})")
print(f"\nTrain/Test ratio: {len(train_df)/len(test_df):.2f}")

# Verify temporal order is preserved
print(f"\nUnique sequences in train: {train_df.groupby(['Driver', 'Race', 'Year']).ngroups}")
print(f"Unique sequences in test: {test_df.groupby(['Driver', 'Race', 'Year']).ngroups}")

In [9]:
train_df = train_df.drop(['Driver', 'Race'], axis=1)

In [10]:
# Convert Compound column to categorical with specified order
categories = ['SOFT', 'MEDIUM', 'HARD']
dtype = pd.CategoricalDtype(categories=categories, ordered=True)
train_df['Compound'] = train_df['Compound'].astype(dtype)

# Verify the conversion
train_df['Compound'].cat.categories.tolist()
train_df['Compound'].cat.codes.value_counts()

In [11]:
# Impute missing values
imputer = SimpleImputer(strategy='most_frequent')
train_df['Compound'] = imputer.fit_transform(train_df['Compound'].values.reshape(-1, 1)).flatten()

In [19]:
# Encode Compound column with OrdinalEncoder
encoder = OrdinalEncoder()
train_df['Compound_encoded'] = encoder.fit_transform(train_df['Compound'].to_numpy().reshape(-1, 1)).flatten()

# Drop the original Compound column
train_df.drop('Compound', axis=1, inplace=True)

In [ ]:
train_df